In [1]:

from itertools import product

# Evaluate propositional logic expressions
def pl_true(sentence, model):

    if isinstance(sentence, bool):
        return sentence

    op = sentence[0]

    if op == 'NOT':
        return not pl_true(sentence[1], model)

    elif op == 'AND':
        return all(pl_true(s, model) for s in sentence[1:])

    elif op == 'OR':
        return any(pl_true(s, model) for s in sentence[1:])

    elif op == 'IMPLIES':
        return (not pl_true(sentence[1], model)
                or pl_true(sentence[2], model))

    elif op == 'IFF':
        return (pl_true(sentence[1], model)
                == pl_true(sentence[2], model))

    else:
        return model[sentence]


# Truth-table entailment algorithm
def tt_check_all(KB, alpha, symbols, model):

    if not symbols:
        if pl_true(KB, model):
            return pl_true(alpha, model)
        else:
            return True

    P = symbols[0]
    rest = symbols[1:]

    return (
        tt_check_all(KB, alpha, rest,
                     {**model, P: True})
        and
        tt_check_all(KB, alpha, rest,
                     {**model, P: False})
    )


def tt_entails(KB, alpha):

    symbols = ['P', 'Q', 'R']

    return tt_check_all(KB, alpha, symbols, {})


# Given Knowledge Base
KB = (
    'AND',
    ('IMPLIES', 'Q', 'P'),
    ('IMPLIES', 'P', ('NOT', 'Q')),
    ('OR', 'Q', 'R')
)

# Generate truth table
print("P Q R | Q->P | P->~Q | QvR | KB")

for p, q, r in product([True, False], repeat=3):

    model = {'P': p, 'Q': q, 'R': r}

    a = pl_true(('IMPLIES', 'Q', 'P'), model)
    b = pl_true(('IMPLIES', 'P', ('NOT', 'Q')), model)
    c = pl_true(('OR', 'Q', 'R'), model)
    kb_value = pl_true(KB, model)

    print(
        p, q, r, "|",
        a, "|", b, "|", c, "|", kb_value
    )

# Check the four questions
print("\nEntailment Results:")

print("KB entails R:",
      tt_entails(KB, 'R'))

print("KB entails R -> P:",
      tt_entails(KB, ('IMPLIES', 'R', 'P')))

print("KB entails Q -> R:",
      tt_entails(KB, ('IMPLIES', 'Q', 'R')))


P Q R | Q->P | P->~Q | QvR | KB
True True True | True | False | True | False
True True False | True | False | True | False
True False True | True | True | True | True
True False False | True | True | False | False
False True True | False | True | True | False
False True False | False | True | True | False
False False True | True | True | True | True
False False False | True | True | False | False

Entailment Results:
KB entails R: True
KB entails R -> P: False
KB entails Q -> R: True
